# CoSense artifact load/inference smoke test
Attach the saved outputs of the B1 and B2 NN notebooks as inputs. This notebook extracts each model ZIP, loads the checkpoint and tokenizer offline, runs one CPU forward pass, and writes `ARTIFACT_SMOKE_TEST.json`.

In [ ]:
import gc, json, math, zipfile
from pathlib import Path
import torch
from torch import nn
from transformers import AutoConfig, AutoModel, AutoTokenizer

class Regressor(nn.Module):
    def __init__(self, encoder, outputs, variant):
        super().__init__()
        self.encoder = encoder
        self.variant = variant
        self.layer = -1
        self.head = nn.Sequential(nn.Dropout(.1), nn.Linear(encoder.config.hidden_size * 2, outputs))

    @staticmethod
    def pool(hidden, mask):
        mask = mask.unsqueeze(-1).to(hidden.dtype)
        return (hidden * mask).sum(1) / mask.sum(1).clamp_min(1)

    def forward(self, input_ids, attention_mask, part1_mask, part2_mask, context_mask=None):
        hidden = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        pooled = [self.pool(hidden, part1_mask), self.pool(hidden, part2_mask)]
        return self.head(torch.cat(pooled, dim=-1))

def safe_extract(source, destination):
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(source) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(destination.resolve()):
                raise ValueError(f'Unsafe ZIP member: {member.filename}')
        archive.extractall(destination)

def locate(variant):
    matches = list(Path('/kaggle/input').rglob(f'NN_{variant}_seed42_full_model.zip'))
    if len(matches) != 1:
        raise FileNotFoundError(f'Expected one {variant} model ZIP, found {matches}')
    return matches[0]


In [ ]:
def smoke(variant):
    archive = locate(variant)
    root = Path('/kaggle/working') / f'{variant}_artifact'
    safe_extract(archive, root)
    metadata = json.loads((root / 'run_metadata.json').read_text())
    config = json.loads((root / 'config.json').read_text())
    assert metadata['model'] == variant and metadata['task'] == 'nn' and metadata['seed'] == 42
    tokenizer = AutoTokenizer.from_pretrained(root / 'tokenizer', use_fast=True, local_files_only=True)
    encoder_config = AutoConfig.from_pretrained(root / 'encoder_config', local_files_only=True)
    encoder = AutoModel.from_config(encoder_config)
    model = Regressor(encoder, outputs=2, variant=variant)
    checkpoint = torch.load(root / 'best.pt', map_location='cpu', weights_only=False)
    model.load_state_dict(checkpoint['model'], strict=True)
    model.eval()
    words = ['flea', 'market'] if variant == 'B1' else ['The', 'flea', 'market', 'opened', 'today', '.']
    encoded = tokenizer(words, is_split_into_words=True, return_tensors='pt', padding='max_length', truncation=True, max_length=config['max_length'])
    word_ids = encoded.word_ids(0)
    first, second = (0, 1) if variant == 'B1' else (1, 2)
    part1 = torch.tensor([[int(value == first) for value in word_ids]])
    part2 = torch.tensor([[int(value == second) for value in word_ids]])
    context = torch.tensor([[int(value is not None and value not in (first, second)) for value in word_ids]])
    with torch.inference_mode():
        prediction = model(encoded['input_ids'], encoded['attention_mask'], part1, part2, context).squeeze(0)
    values = [float(value) for value in prediction]
    if len(values) != 2 or not all(math.isfinite(value) for value in values):
        raise ValueError(f'Invalid {variant} prediction: {values}')
    result = {'status': 'PASS', 'model': variant, 'task': 'nn', 'seed': 42, 'checkpoint_epoch_index': int(checkpoint['epoch']), 'prediction': values, 'archive': archive.name}
    del model, encoder, checkpoint
    gc.collect()
    return result

results = [smoke('B1'), smoke('B2')]
report = {'status': 'PASS', 'checks': results}
Path('/kaggle/working/ARTIFACT_SMOKE_TEST.json').write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))
